# Gather Full Text

This notebook retrieves the full text of articles from the PMC (PubMed Central) API. The process involves creating a mapping between PMIDs (PubMed IDs) and their corresponding full text documents.

## Approach

1. **Load or Build Mapping**: The notebook first checks if a cached PMID-to-full-text mapping already exists. If it does, the mapping is loaded from disk to avoid redundant API calls. If not, the mapping is built by querying the PMC dataset.

2. **Note on Computational Requirements**: Building this mapping from scratch is computationally intensive and requires substantial RAM, as the entire PMC dataset must be loaded into memory. For large-scale implementations on your own datasets, consider refactoring this code to stream through the API instead of loading everything at once.

3. **Data Filtering**: After the mapping is created, the dataframe is filtered to retain only articles that have full text available in PMC, removing entries without corresponding full text documents.

In [1]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import pickle
import brotli
import os

In [2]:
df_references = pd.read_parquet("../data/00_references/merged_references.parquet.br")

In [3]:
# here we build a mapping from PMID to full text, which is needed to 
# match the references in our dataset to their corresponding full 
# texts in the PMC Open Access dataset. We save this mapping as a 
# compressed pickle file for faster loading in the future, since 
# building it from scratch can be time-consuming.

mapping_path = "../data/01_full_text/pmid_to_full_text.pkl.br"
pmids_needed = set(df_references["PMID"].astype(str).tolist())

if os.path.exists(mapping_path):
    print(f"Loading compressed PMID → full_text mapping from {mapping_path}")
    with open(mapping_path, "rb") as f:
        compressed_data = f.read()
        pmid_to_full_text = pickle.loads(brotli.decompress(compressed_data))
else:
    print("Mapping file not found. Building PMID → full_text mapping...")

    # Note that this dataset requires trust_remote_code=True, which is only available for datasets<4.0.0
    pmc_ds = load_dataset("pmc/open_access", split="train", trust_remote_code=True)

    pmid_to_full_text = {
        row["pmid"]: row["text"]
        for row in tqdm(pmc_ds, total=len(pmc_ds), desc="Building PMID → full_text mapping")
        if row["pmid"] in pmids_needed
    }

    compressed_data = brotli.compress(
        pickle.dumps(pmid_to_full_text, protocol=pickle.HIGHEST_PROTOCOL),
        quality=11  # max compression
    )

    with open(mapping_path, "wb") as f:
        f.write(compressed_data)

    print(f"Compressed mapping saved to {mapping_path}")

Loading compressed PMID → full_text mapping from ../data/01_full_text/pmid_to_full_text.pkl.br


In [4]:
df_references["full_text"] = df_references["PMID"].astype(str).map(pmid_to_full_text)

In [5]:
# We only keep the rows that are available in PMC.
print(f"Number of references before filtering for full text: {df_references.shape[0]}")
df_references = df_references[df_references["full_text"].notnull()].reset_index(drop=True)
print(f"Number of references after filtering for full text: {df_references.shape[0]}")

Number of references before filtering for full text: 6658
Number of references after filtering for full text: 5251


In [6]:
df_references.to_parquet("../data/01_full_text/references_with_full_text.parquet.br", index=False, compression="brotli")

In [7]:
df_references.head()

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,full_text
0,35799302,Development and validation of a multivariable ...,"Iachkine J, Buetti N, de Grooth HJ, Briant AR,...",Crit Care. 2022 Jul 7;26(1):205. doi: 10.1186/...,Iachkine J,Crit Care,2022,2022/07/07,PMC9261073,None,10.1186/s13054-022-04078-x,==== Front\nCrit Care\nCritical Care\n1364-853...
1,40151494,Development and validation of a risk-predictio...,"Qian J, Ruan C, Cai Y, Yi W, Liu J, Xu R.",Ther Adv Drug Saf. 2025 Mar 22;16:204209862513...,Qian J,Ther Adv Drug Saf,2025,2025/03/28,PMC11946284,None,10.1177/20420986251328687,==== Front\nTher Adv Drug Saf\nTher Adv Drug S...
2,36383295,28-day sepsis mortality prediction model from ...,"Xie Y, Zhuang D, Chen H, Zou S, Chen W, Chen Y.",Eur J Clin Microbiol Infect Dis. 2023 Jan;42(1...,Xie Y,Eur J Clin Microbiol Infect Dis,2023,2022/11/16,PMC9816294,None,10.1007/s10096-022-04517-1,==== Front\nEur J Clin Microbiol Infect Dis\nE...
3,33746422,Development of a multivariable prediction mode...,"Andersen JD, Hangaard S, Buus AAØ, Laursen M, ...",J Orthop. 2021 Mar 11;24:216-221. doi: 10.1016...,Andersen JD,J Orthop,2021,2021/03/22,PMC7961305,None,10.1016/j.jor.2021.03.001,==== Front\nJ Orthop\nJ Orthop\nJournal of Ort...
4,33815175,Prospective Long-Term Cohort Study of Subjects...,"Peralta V, Moreno-Izco L, García de Jalón E, S...",Front Psychiatry. 2021 Mar 19;12:643112. doi: ...,Peralta V,Front Psychiatry,2021,2021/04/05,PMC8017172,None,10.3389/fpsyt.2021.643112,==== Front\nFront Psychiatry\nFront Psychiatry...
